<h3/>
<h1 align="center">Universidad de Buenos Aires (FIUBA)</h1>
<h2 align="center">Especialización en Inteligencia Artificial - CEIA - FIUBA</h2>
<h2 align="center">Aprendizaje de Máquina I</h2>
<h3 align="center">Predicción de la recaudación de una película</h3>

<p align="left">
Integrantes:
  <ul>
    <li> Luis Jose Paredes Ramirez </li>
    <li> Hernando Scheidl </li>
    <li> Nicolás Lastra </li>
    <li> Diego Vazquez </li>
  <ul/>
</p>

---

# Planteo del problema y Fuente de datos

Utilizamos el dataset de Kaggle [Full TMDB Movies Dataset 2024 (1M Movies)](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies) que contiene informacion sobre peliculas extraidas a partir de la API pública de [The Movie Database (TMDB)](https://developer.themoviedb.org/docs/getting-started). 

Se entrena un modelo de regresion con una variable objetivo `revenue` de una película a partir de sus características. Este target es continua, positiva y extremadamente asimétrica por naturaleza.


Ver [Notebook de EDA](./EDA.ipynb) para ver el analisis del planteo del problema.

---
## 1. Descarga del dataset

In [ ]:
!pip install -q kagglehub optuna xgboost scikit-learn mlflow==3.16.1 boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 22.7 MB/s eta 0:00:00


In [ ]:
import kagglehub
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from kagglehub import KaggleDatasetAdapter
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from xgboost import XGBRegressor


Descargamos el archivo usando el SDK de [`kagglehub`](https://pypi.org/project/kagglehub/). Recordar que `file_path` apunta al nombre del archivo que se descarga.

Establecemos ciertos parametros para utilizarlos durante la notebook y descargamos el dataset.

In [ ]:
sns.set()

# Paleta de colores usada en la notebook
AZUL, MAGENTA, VERDE, NARANJA, CELESTE, GRIS = (
    "#007aff", "#ff48fd", "#44c57f", "#ffa600", "#8ecae6", "#555555",
)

#Definimos numero semilla para las funciones pseudo aleatorias.
RANDOM_STATE = 42

# Se define la ruta de la carpeta donde esta el dataset.
file_path = "TMDB_movie_dataset_v11.csv"

# Descargamos el dataset.
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "asaniczka/tmdb-movies-dataset-2023-930k-movies",
  file_path
)

100%|██████████| 623M/623M [00:04<00:00, 160MB/s]


# ETL

Nos quedamos con las features relevantes y hacemos un pre-procesamiento

In [ ]:
CANDIDATAS = [
    "id", "title", "status", "release_date",   # identificación y filtrado
    "revenue",                                 # nuestra variable objetivo o target
    "budget", "runtime", "popularity",         # numéricas principales
    "vote_average", "vote_count",              # recepción del público
    "original_language",                       # categórica
    "genres", "production_companies",          # listas separadas por coma
]

print(f"Columnas descartadas {set(df.columns) - set(CANDIDATAS)}")

df = df[CANDIDATAS].copy()
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")

Columnas descartadas {'keywords', 'production_countries', 'original_title', 'homepage', 'overview', 'backdrop_path', 'imdb_id', 'tagline', 'spoken_languages', 'adult', 'poster_path'}
Dimensiones: 1,480,412 filas × 13 columnas
Memoria: 508 MB


,id,title,status,release_date,revenue,budget,runtime,popularity,vote_average,vote_count,original_language,genres,production_companies
0,27205,Inception,Released,2010-07-15,825532764,160000000,148,83.952,8.364,34495,en,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict..."
1,157336,Interstellar,Released,2014-11-05,701729206,165000000,169,140.241,8.417,32571,en,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc..."
2,155,The Dark Knight,Released,2008-07-16,1004558444,185000000,152,130.643,8.512,30619,en,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel..."
3,19995,Avatar,Released,2009-12-15,2923706026,237000000,162,79.932,7.573,29815,en,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ..."
4,24428,The Avengers,Released,2012-04-25,1518815515,220000000,143,98.082,7.710,29166,en,"Science Fiction, Action, Adventure",Marvel Studios


---
## 4. Filtrado: del crudo al dataset de modelado

Antes de aplicar los filtros evaluamos cuantos datos vamos filtrando acumulativamente


Nos quedamos con las filas que cumplan:

- **`status == 'Released'`**: Peliculas lanzadas.
- **`revenue > 0` y `budget > 0`**: Datos financieros validos
- **`runtime > 0`**: descarta titulos sin duración cargada.
- **`vote_count > 0` & `vote_average > 0`**: `vote_count` & `vote_average` deben ser valores mayores que 0
- **Fecha de estreno y género no nulos**: sacamos los valores nulos

In [ ]:
filtros = [
    ("Dataset completo",              pd.Series(True, index=df.index)),
    ("status == 'Released'",          df["status"].eq("Released")),
    ("revenue > 0",                   df["revenue"] > 0),
    ("budget  > 0",                   df["budget"] > 0),
    ("runtime > 0",                   df["runtime"] > 0),
    ("vote_count > 0",                df["vote_count"] > 0),
    ("vote_average > 0",              df["vote_average"] > 0),
    ("fecha not null",        df["release_date"].notna()),
    ("género not null",          df["genres"].notna()),
]

mascara = pd.Series(True, index=df.index)
traza = []
for nombre, condicion in filtros:
    mascara &= condicion.fillna(False)
    traza.append({"filtro": nombre,
                  "filas": int(mascara.sum()),
                  "% del original": mascara.mean() * 100})

embudo = pd.DataFrame(traza).set_index("filtro").round(2)

,filas,% del original
filtro,,
Dataset completo,1480412,100.00
status == 'Released',1424043,96.19
revenue > 0,24781,1.67
budget > 0,17787,1.20
runtime > 0,14880,1.01
vote_count > 0,10773,0.73
vote_average > 0,10765,0.73
fecha not null,10593,0.72
género not null,10515,0.71


In [ ]:
peliculas = df[mascara].drop_duplicates(subset="id").reset_index(drop=True)

def parsear_lista(valor):
    """Convierte 'Drama, Crimen' en ['Drama', 'Crimen']."""
    if pd.isna(valor):
        return []
    return [x.strip() for x in str(valor).split(",") if x.strip()]


generos_expandidos = peliculas["genres"].apply(parsear_lista).explode().dropna()
conteo_generos = generos_expandidos.value_counts()


TOP_GENEROS = conteo_generos.head(12).index.tolist()
TOP_IDIOMAS = peliculas["original_language"].value_counts().head(8).index.tolist()

Dataset de modelado: 10,515 películas (0.71% del original)
Años cubiertos: 1913 – 2026


In [ ]:
def construir_features(datos):
    """Arma la matriz de diseño a partir del dataframe filtrado."""
    listas_generos = datos["genres"].apply(parsear_lista)

    matriz = pd.DataFrame(index=datos.index)

    # Sesgadas (se les aplica log en el pipeline)
    matriz["budget"] = datos["budget"]
    matriz["popularity"] = datos["popularity"]
    matriz["vote_count"] = datos["vote_count"]
    matriz["n_companias"] = (datos["production_companies"]
                             .apply(parsear_lista).str.len())

    # Numéricas
    matriz["runtime"] = datos["runtime"]
    matriz["vote_average"] = datos["vote_average"]
    matriz["anio"] = datos["release_date"].dt.year

    # Categórica: los idiomas poco frecuentes se agrupan en "otro"
    matriz["idioma"] = datos["original_language"].where(
        datos["original_language"].isin(TOP_IDIOMAS), "otro")

    # Binarias: un indicador por género (multietiqueta)
    for genero in TOP_GENEROS:
        columna = "gen_" + genero.lower().replace(" ", "_")
        matriz[columna] = listas_generos.apply(lambda gs, g=genero: int(g in gs))

    return matriz


SESGADAS = ["budget", "popularity", "vote_count", "n_companias"]
NUMERICAS = ["runtime", "vote_average", "anio"]
CATEGORICAS = ["idioma"]
BINARIAS = ["gen_" + g.lower().replace(" ", "_") for g in TOP_GENEROS]

X = construir_features(peliculas)
y = peliculas["revenue"]

print(f"Matriz de features: {X.shape[0]:,} filas × {X.shape[1]} columnas")
X.head()

Matriz de features: 10,515 filas × 20 columnas


,budget,popularity,vote_count,n_companias,runtime,vote_average,anio,idioma,gen_drama,gen_comedy,gen_action,gen_thriller,gen_romance,gen_adventure,gen_crime,gen_horror,gen_family,gen_science_fiction,gen_fantasy,gen_mystery
0,160000000,83.952,34495,3,148,8.364,2010,en,0,0,1,0,0,1,0,0,0,1,0,0
1,165000000,140.241,32571,3,169,8.417,2014,en,1,0,0,0,0,1,0,0,0,1,0,0
2,185000000,130.643,30619,5,152,8.512,2008,en,1,0,1,1,0,0,1,0,0,0,0,0
3,237000000,79.932,29815,4,162,7.573,2009,en,0,0,1,0,0,1,0,0,0,1,1,0
4,220000000,98.082,29166,1,143,7.710,2012,en,0,0,1,0,0,1,0,0,0,1,0,0


### Nulos

Verificamos que no queden valores faltantes antes de entrenar. Los filtros de la sección 4 ya
descartaron implícitamente casi todos, pero validamos por las dudas. Descartamos las filas incompletas en lugar de imputarlas.

In [ ]:
nulos_por_columna = X.isna().sum()
print("Nulos por columna:")
print(nulos_por_columna[nulos_por_columna > 0]
      if nulos_por_columna.any() else "  (ninguno)")

completas = X.notna().all(axis=1) & y.notna()

X, y = X[completas], y[completas]
print(f"\nFilas descartadas por nulos: {(~completas).sum()}")
print(f"Dataset final: {len(X):,} películas × {X.shape[1]} features")

Nulos por columna:
  (ninguno)

Filas descartadas por nulos: 0
Dataset final: 10,515 películas × 20 features


Hacemos el split para obtener nuestro conjunto de entrenamiento y nuestro conjunto de evaluacion

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

print(f"Entrenamiento: {len(X_train):,} películas")
print(f"Testeo:        {len(X_test):,} películas")

Entrenamiento: 8,412 películas
Testeo:        2,103 películas


# Entrenar

Del [Notebook EDA](./EDA.ipynb) vemos que el mejor modelo es XGBoost (con hiperparametros optimizados)

Entrenamos directamente ese modelo

In [ ]:
preprocesador = ColumnTransformer([
    # log + estandarización para las variables de muchos órdenes de magnitud
    ("sesgadas", Pipeline([("log", FunctionTransformer(np.log1p)),
                           ("escala", StandardScaler())]), SESGADAS),
    # estandarización a secas para el resto de las numéricas
    ("numericas", StandardScaler(), NUMERICAS),
    # one-hot para la categórica
    ("categoricas", OneHotEncoder(handle_unknown="ignore",
                                  sparse_output=False), CATEGORICAS),
    # los indicadores de género ya son 0/1
    ("binarias", "passthrough", BINARIAS),
])


def pipeline(estimador):
    """Preprocesamiento + modelo, con el target en escala logarítmica."""
    return TransformedTargetRegressor(
        regressor=Pipeline([("prep", preprocesador),
                            ("modelo", clone(estimador))]),
        func=np.log1p,
        inverse_func=np.expm1,
    )


def objetivo(trial):
    """MAE por validación cruzada de una configuración propuesta por Optuna."""
    candidato = XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 100, 1000, step=50),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.4, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 15),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        random_state=RANDOM_STATE, n_jobs=1,
    )
    return -cross_val_score(pipeline(candidato), 
                            X_train, y_train, cv=5,
                            scoring="neg_mean_absolute_error", 
                            n_jobs=-1).mean()

In [ ]:
N_TRIALS = 150   # la clase 5 usa 300; bajarlo o subirlo solo cambia el tiempo

optuna.logging.set_verbosity(optuna.logging.ERROR)

estudio = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))

estudio.optimize(objetivo, n_trials=N_TRIALS, show_progress_bar=False)

best_model = pipeline(XGBRegressor(**estudio.best_params,
                                   random_state=RANDOM_STATE, n_jobs=-1)
                    )